# 0.1 Import Libraries

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# 0.2 Load Project Modules

In [ ]:
PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from importlib import reload
from config.settings import PROCESSED_DIR, OUTPUT_DIR
import src.panel_health as panel_health

panel_health = reload(panel_health)
from src.panel_health import active_store_rate, sample_coverage_score, category_coverage_score, top_store_concentration_score, extrapolation_reliability_score, panel_stability_score, overall_panel_health_score, assign_risk_level, recommended_action, build_panel_health_summary

# 1.1 Load Universe Outputs

In [ ]:
universe = pd.read_csv(OUTPUT_DIR / "active_store_universe.csv")
universe.head()

# 1.2 Load Sample Panel Outputs

In [ ]:
panel_store_list = pd.read_csv(OUTPUT_DIR / "sample_panel_store_list.csv")
if "is_active" not in panel_store_list.columns:
    panel_store_list = panel_store_list.merge(
        universe[["store_nbr", "is_active"]],
        on="store_nbr",
        how="left",
    )
panel = panel_store_list[panel_store_list["panel_name"] == "optimized_panel"].copy()
panel.head()

# 1.3 Load Extrapolation Outputs

In [ ]:
extrapolation_comparison = pd.read_csv(OUTPUT_DIR / "extrapolation_method_comparison.csv")
weekly_error = pd.read_csv(OUTPUT_DIR / "weekly_extrapolation_error.csv")
category_error = pd.read_csv(OUTPUT_DIR / "category_extrapolation_error.csv")
extrapolation_comparison.head()

# 1.4 Load Missing Retailer Outputs

In [ ]:
missing_retailer_impact = pd.read_csv(OUTPUT_DIR / "missing_retailer_impact_summary.csv")
weekly_category_sales = pd.read_csv(PROCESSED_DIR / "weekly_store_category_sales.csv", parse_dates=["week"])
missing_retailer_impact.head()

# 2.1 Calculate Active Store Rate

In [ ]:
active_rate = active_store_rate(panel)
active_rate

# 2.2 Calculate Sample Coverage Score

In [ ]:
sample_score = sample_coverage_score(panel, universe)
sample_score

# 2.3 Calculate Category Coverage Score

In [ ]:
category_score = category_coverage_score(panel, weekly_category_sales)
category_score

# 2.4 Calculate Store Contribution Concentration

In [ ]:
top_concentration = top_store_concentration_score(panel)
top_concentration

# 2.5 Calculate Extrapolation Reliability Score

In [ ]:
best_error_pct = extrapolation_comparison.sort_values("wape")["bias_pct"].iloc[0]
reliability_score = extrapolation_reliability_score(best_error_pct)
reliability_score

# 2.6 Calculate Panel Stability Score

In [ ]:
stability_score = panel_stability_score(panel)
stability_score

# 3.1 Build Overall Panel Health Score

In [ ]:
overall_score = overall_panel_health_score(sample_score, category_score, top_concentration, reliability_score, stability_score)
overall_score

# 3.2 Assign Risk Level

In [ ]:
risk_level = assign_risk_level(overall_score)
risk_level

# 3.3 Generate Recommended Actions

In [ ]:
panel_health = build_panel_health_summary(panel_store_list, universe, weekly_category_sales, extrapolation_comparison)
panel_health["recommended_action"]

# 4.1 Save Panel Health Outputs

In [ ]:
panel_health.to_csv(OUTPUT_DIR / "panel_health_kpi_summary.csv", index=False)
dashboard_summary = pd.DataFrame({
    "metric": ["active_store_count", "panel_count", "best_extrapolation_method", "lowest_wape", "highest_missing_retailer_risk"],
    "value": [
        universe["store_nbr"].nunique(),
        panel_store_list["panel_name"].nunique(),
        extrapolation_comparison.sort_values("wape")["method"].iloc[0],
        extrapolation_comparison["wape"].min(),
        missing_retailer_impact["business_risk_level"].iloc[0] if not missing_retailer_impact.empty else "unknown",
    ],
})
dashboard_summary.to_csv(OUTPUT_DIR / "dashboard_summary.csv", index=False)
panel_health